# Contextual visible-text baseline

Colab workflow for E3 in `IMPROVEMENTS.md`. It clones the private repository, authenticates to Hugging Face, runs the frozen contextual text baseline, stores large run artifacts on Google Drive, and can push a small Markdown result record back to GitHub.

Before running, add these Colab secrets (not notebook variables): `GITHUB_TOKEN` with repository Contents read/write permission and `HF_TOKEN` with Hugging Face read permission. The notebook never prints either token. `PUSH_RESULTS` is `False` by default.

## Authenticate and clone

The repository is cloned into the ephemeral runtime. The dataset, activation metadata, logs, and baseline artifacts live on Drive so the run can resume after a disconnect.

In [ ]:
import os
import shutil
import stat
import subprocess
import sys
from contextlib import contextmanager
from pathlib import Path

from google.colab import drive, userdata

REPOSITORY_URL = "https://github.com/sagnikc395/tracing-math.git"
REPOSITORY_BRANCH = "main"
GIT_NAME = "sagnikc395"
GIT_EMAIL = "sagnikchatterjee607@gmail.com"

def required_secret(name):
    try:
        value = userdata.get(name)
    except Exception as error:
        raise RuntimeError(f"Add the Colab secret {name!r} and grant notebook access.") from error
    if not value:
        raise RuntimeError(f"Colab secret {name!r} is empty or unavailable.")
    return value

GITHUB_TOKEN = required_secret("GITHUB_TOKEN")
HF_TOKEN = required_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ.setdefault("HF_HOME", "/content/huggingface")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

@contextmanager
def github_auth():
    askpass = Path("/content/github-askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf '%s\\n' x-access-token ;;\n"
        "  *) printf '%s\\n' \"$GITHUB_TOKEN\" ;;\n"
        "esac\n"
    )
    askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
    environment = os.environ.copy()
    environment.update({"GITHUB_TOKEN": GITHUB_TOKEN, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    try:
        yield environment
    finally:
        askpass.unlink(missing_ok=True)

PROJECT_ROOT = Path("/content/tracing-math")
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
with github_auth() as git_environment:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
        env=git_environment,
    )
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)], check=True)
subprocess.run(["git", "config", "user.name", GIT_NAME], cwd=PROJECT_ROOT, check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], cwd=PROJECT_ROOT, check=True)
print(f"Cloned {REPOSITORY_URL} at {PROJECT_ROOT}")


In [ ]:
import torch
from huggingface_hub import login

login(token=HF_TOKEN, add_to_git_credential=False)
if not torch.cuda.is_available():
    raise RuntimeError("Use a Colab GPU runtime for contextual encoding.")
print(f"GPU: {torch.cuda.get_device_name(0)}")


## Resolve the run configuration

The scientific settings come from `configs/project.yaml`. This cell changes only runtime paths. Do not change the encoder, split seed, C grid, or evaluation protocol after inspecting the held-out result.

In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd
import yaml
from IPython.display import display

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/math-error-tracing")
RUN_ROOT = DRIVE_ROOT / "contextual-baseline"
DATA_PATH = DRIVE_ROOT / "data/processbench.jsonl"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts/qwen2.5-math-1.5b-a100-bf16"
ANALYSIS_ROOT = RUN_ROOT / "artifacts/analysis"
for path in (DATA_PATH.parent, ARTIFACT_ROOT, ANALYSIS_ROOT):
    path.mkdir(parents=True, exist_ok=True)

base_config = yaml.safe_load((PROJECT_ROOT / "configs/project.yaml").read_text())
analysis_config = base_config.setdefault("analysis", {})
# Backward-compatible defaults for a checkout made before the E3 config fields existed.
analysis_config.setdefault("contextual_encoder_name", "sentence-transformers/all-MiniLM-L6-v2")
analysis_config.setdefault("contextual_encoder_revision", None)
analysis_config.setdefault("contextual_batch_size", 16)
analysis_config.setdefault("contextual_max_length", 512)
base_config["data"]["output_path"] = str(DATA_PATH)
base_config["extraction"]["output_dir"] = str(ARTIFACT_ROOT)
base_config["artifacts"]["analysis_dir"] = str(ANALYSIS_ROOT)
CONFIG_PATH = Path("/content/contextual_baseline_project.yaml")
CONFIG_PATH.write_text(yaml.safe_dump(base_config, sort_keys=False))
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"Run root: {RUN_ROOT}")
print(f"Activation metadata: {ARTIFACT_ROOT / 'activation_shards'}")
print(f"Contextual encoder: {analysis_config['contextual_encoder_name']}")


## Prerequisites

`fit-contextual-baseline` needs the normalized ProcessBench file and the primary activation shard metadata. It does not load target-model hidden states. Set `RUN_PRIMARY_EXTRACTION = True` only when those files are not already on Drive; that stage downloads data and runs the target model.

In [ ]:
def run_cli(*arguments):
    command = [sys.executable, "-m", "tracing_math", "--config", str(CONFIG_PATH), *arguments]
    print("$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

RUN_PRIMARY_EXTRACTION = False
if RUN_PRIMARY_EXTRACTION:
    run_cli("download-data")
    run_cli("extract-activations")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {DATA_PATH}. Set RUN_PRIMARY_EXTRACTION = True.")
if not (ARTIFACT_ROOT / "activation_shards").exists():
    raise FileNotFoundError("Missing activation shards. Set RUN_PRIMARY_EXTRACTION = True.")
run_cli("validate-config")


## Run the frozen contextual baseline

The command encodes the problem and text through each current boundary, then fits a trace-equal logistic head. It selects C by validation AUROC and the operating threshold by validation Process F1.

In [ ]:
available_commands = subprocess.run(
    [sys.executable, "-m", "tracing_math", "--help"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout
if "fit-contextual-baseline" not in available_commands:
    raise RuntimeError(
        "The cloned branch does not include fit-contextual-baseline. Push the E3 implementation "
        "(contextual.py, config, CLI, and operations changes) to the selected GitHub branch, then "
        "rerun the Authenticate and clone cell."
    )
run_cli("fit-contextual-baseline")

CONTEXTUAL_ROOT = ANALYSIS_ROOT / "contextual_text_baseline"
metrics = pd.read_csv(CONTEXTUAL_ROOT / "metrics.csv")
selection = pd.read_csv(CONTEXTUAL_ROOT / "validation_selection.csv")
predictions = pd.read_csv(CONTEXTUAL_ROOT / "test_predictions.csv")
resolved = json.loads((CONTEXTUAL_ROOT / "resolved_config.json").read_text())
display(metrics.round(4))
display(selection.round(4))
print(json.dumps(resolved, indent=2))
print(f"Held-out boundaries: {len(predictions)}; traces: {predictions.trace_id.nunique()}")


## Create and optionally push a result record

Only the generated Markdown record is staged. Leave `PUSH_RESULTS = False` to inspect the result first. A successful push does not upload Drive artifacts, activation shards, embeddings, datasets, or credentials.

In [ ]:
PUSH_RESULTS = False

record_path = PROJECT_ROOT / "results" / f"contextual_baseline_{RUN_TAG}.md"
metric_row = metrics.iloc[0]
selected = selection.loc[selection.selected].iloc[0]
record_path.write_text(
    "# Contextual visible-text baseline\n\n"
    f"Run: `{RUN_TAG}`\n\n"
    "This is an exploratory E3 result. The encoder saw only the problem and prefix through the current boundary.\n\n"
    f"- Encoder: `{resolved['encoder']}`\n"
    f"- Encoder revision: `{resolved['revision']}`\n"
    f"- Selected C: `{selected.c_value}`\n"
    f"- Validation AUROC at selected C: `{selected.validation_auroc:.4f}`\n"
    f"- Held-out AUROC: `{metric_row.auroc:.4f}`\n"
    f"- Held-out average precision: `{metric_row.average_precision:.4f}`\n"
    f"- Held-out log loss: `{metric_row.log_loss:.4f}`\n"
    f"- Held-out Process F1: `{metric_row.process_f1:.4f}`\n"
    f"- Held-out traces: `{predictions.trace_id.nunique()}`\n"
    f"- Resolved configuration SHA-256: `{resolved['sha256']}`\n"
 )
print(record_path)

if PUSH_RESULTS:
    subprocess.run(["git", "add", "--", str(record_path.relative_to(PROJECT_ROOT))], cwd=PROJECT_ROOT, check=True)
    staged = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=PROJECT_ROOT).returncode
    if staged == 0:
        print("No result-record change to commit.")
    else:
        subprocess.run(
            ["git", "commit", "-m", "results: add contextual baseline record"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        with github_auth() as git_environment:
            subprocess.run(["git", "push", "origin", REPOSITORY_BRANCH], cwd=PROJECT_ROOT, check=True, env=git_environment)
        print("Pushed the result record to GitHub.")
else:
    print("Result record created locally. Inspect it, then set PUSH_RESULTS = True to commit and push it.")
